In [3]:
# ============================================================================
# 🎓 OXFORD UNIVERSITY PhD-LEVEL ANALYSIS
# COVID-19 VACCINE SIDE EFFECTS PREDICTION
# Complete End-to-End Systematic Reconstruction
# ============================================================================

"""
METHODOLOGY OVERVIEW (নীতিগত কাঠামো):

STEP 1: Data Loading & Cleaning
   ├─ Load raw dataset
   ├─ Column name standardization
   ├─ Missing value handling
   └─ Data quality check

STEP 2: Feature Selection (Traditional Methods)
   ├─ Chi-Square Test (categorical association)
   ├─ Mutual Information (information gain)
   ├─ Random Forest Importance (tree-based)
   ├─ Boruta Algorithm (ensemble selection)
   └─ Consensus Voting (≥2 methods)

STEP 3: Novel MAFS Algorithm
   ├─ Multi-Stage Adaptive Feature Selection
   ├─ Stage 1: Variance + Correlation filtering
   ├─ Stage 2: Statistical significance testing
   ├─ Stage 3: ML importance with stability
   ├─ Stage 4: COVID-domain knowledge integration
   └─ Stage 5: Ensemble consensus

STEP 4: Feature Set Decision
   ├─ Compare traditional vs MAFS
   ├─ Determine optimal combination
   ├─ Final feature justification
   └─ Document selection rationale

STEP 5: Data Preprocessing
   ├─ Train-Test Split (stratified, 80-20)
   ├─ Feature scaling/normalization
   ├─ SMOTE for class imbalance (train only)
   └─ Cross-validation setup

STEP 6: Model Training & Evaluation
   ├─ Train multiple models
   ├─ Cross-validation analysis
   ├─ Hyperparameter tuning
   ├─ Best model selection
   └─ Statistical testing

STEP 7: Advanced Analysis (Q1 Publication)
   ├─ ROC/PR curves
   ├─ SHAP interpretability
   ├─ Model calibration
   ├─ Bootstrap confidence
   └─ Clinical validation
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix)
from imblearn.over_sampling import SMOTE
from boruta import BorutaPy
import warnings
warnings.filterwarnings('ignore')

# Set global random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("="*80)
print("🎓 OXFORD UNIVERSITY PhD-LEVEL ANALYSIS")
print("COVID-19 VACCINE SIDE EFFECTS PREDICTION - CLEAN SYSTEMATIC CODE")
print("="*80)

# ============================================================================
# STEP 1: DATA LOADING & CLEANING
# ============================================================================

print("\n" + "="*80)
print("STEP 1: DATA LOADING & CLEANING")
print("="*80)

# Load dataset
df_raw = pd.read_csv('featureselection code.csv')
print(f"✅ Loaded raw data: {df_raw.shape}")

# Standardize column names
df = df_raw.copy()
df.columns = df.columns.str.strip().str.replace('\n', '').str.replace(' ', '_')
print(f"✅ Standardized {len(df.columns)} column names")

# Define target variable
TARGET = 'Side_effects_of__COVID-19_vaccine'
print(f"✅ Target variable: {TARGET}")

# Drop non-numeric columns (except target)
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
if TARGET not in numeric_cols and TARGET in df.columns:
    numeric_cols.append(TARGET)

df_clean = df[numeric_cols].copy()
print(f"✅ Retained {len(df_clean.columns)} numeric features + target")

# Check missing values
missing_count = df_clean.isnull().sum().sum()
print(f"✅ Missing values: {missing_count}")

# Prepare features and target
X_full = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

print(f"✅ Features shape: {X_full.shape}")
print(f"✅ Target shape: {y.shape}")
print(f"✅ Target distribution: {y.value_counts().to_dict()}")

# ============================================================================
# STEP 2: TRADITIONAL FEATURE SELECTION
# ============================================================================

print("\n" + "="*80)
print("STEP 2: TRADITIONAL FEATURE SELECTION")
print("="*80)

# Scale features for Chi-square test
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_full)

print("\n📊 APPLYING FEATURE SELECTION METHODS:")

# 2.1 Chi-Square Test
print("\n1️⃣ CHI-SQUARE TEST:")
chi2_selector = SelectKBest(score_func=chi2, k='all')
chi2_selector.fit(X_scaled, y)
chi2_scores = chi2_selector.scores_
print(f"   ✅ Calculated scores for {len(chi2_scores)} features")

# 2.2 Mutual Information
print("\n2️⃣ MUTUAL INFORMATION:")
mi_selector = SelectKBest(score_func=mutual_info_classif, k='all')
mi_selector.fit(X_full, y)
mi_scores = mi_selector.scores_
print(f"   ✅ Calculated scores for {len(mi_scores)} features")

# 2.3 Random Forest Importance
print("\n3️⃣ RANDOM FOREST IMPORTANCE:")
rf_selector = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_selector.fit(X_full, y)
rf_importances = rf_selector.feature_importances_
print(f"   ✅ Calculated importance for {len(rf_importances)} features")

# 2.4 Boruta Algorithm
print("\n4️⃣ BORUTA ALGORITHM:")
try:
    rf_boruta = RandomForestClassifier(n_jobs=-1, class_weight='balanced', 
                                       max_depth=5, random_state=RANDOM_STATE)
    boruta_selector = BorutaPy(rf_boruta, n_estimators='auto', random_state=RANDOM_STATE, verbose=0)
    boruta_selector.fit(X_full.values, y.values)
    boruta_support = boruta_selector.support_
    print(f"   ✅ Boruta selected {sum(boruta_support)} features")
except Exception as e:
    print(f"   ⚠️ Boruta failed: {e}")
    boruta_support = [False] * X_full.shape[1]

# 2.5 Create Feature Selection Summary
feature_selection_df = pd.DataFrame({
    'Feature': X_full.columns,
    'Chi2_Score': chi2_scores,
    'MI_Score': mi_scores,
    'RF_Importance': rf_importances,
    'Boruta_Selected': boruta_support
})

# Calculate thresholds (mean-based)
chi2_threshold = feature_selection_df['Chi2_Score'].mean()
mi_threshold = feature_selection_df['MI_Score'].mean()
rf_threshold = feature_selection_df['RF_Importance'].mean()

# Mark features above threshold
feature_selection_df['Chi2_Pass'] = feature_selection_df['Chi2_Score'] > chi2_threshold
feature_selection_df['MI_Pass'] = feature_selection_df['MI_Score'] > mi_threshold
feature_selection_df['RF_Pass'] = feature_selection_df['RF_Importance'] > rf_threshold
feature_selection_df['Boruta_Pass'] = feature_selection_df['Boruta_Selected']

# Consensus voting (≥2 methods agreement)
feature_selection_df['Vote_Count'] = (
    feature_selection_df['Chi2_Pass'].astype(int) +
    feature_selection_df['MI_Pass'].astype(int) +
    feature_selection_df['RF_Pass'].astype(int) +
    feature_selection_df['Boruta_Pass'].astype(int)
)

traditional_selected = feature_selection_df[feature_selection_df['Vote_Count'] >= 2]['Feature'].tolist()

print(f"\n📋 TRADITIONAL FEATURE SELECTION RESULTS:")
print(f"   Chi2 selected: {feature_selection_df['Chi2_Pass'].sum()} features")
print(f"   MI selected: {feature_selection_df['MI_Pass'].sum()} features")
print(f"   RF selected: {feature_selection_df['RF_Pass'].sum()} features")
print(f"   Boruta selected: {feature_selection_df['Boruta_Pass'].sum()} features")
print(f"   📌 CONSENSUS (≥2 methods): {len(traditional_selected)} features")
print(f"   Features: {traditional_selected}")

# ============================================================================
# STEP 3: NOVEL MAFS ALGORITHM
# ============================================================================

print("\n" + "="*80)
print("STEP 3: NOVEL MULTI-STAGE ADAPTIVE FEATURE SELECTION (MAFS)")
print("="*80)

from scipy.stats import f_classif

def mafs_algorithm(X, y, feature_names):
    """
    Multi-Stage Adaptive Feature Selection (MAFS)
    
    STAGE 1: Variance & Correlation filtering
    STAGE 2: Statistical significance testing
    STAGE 3: ML importance with stability
    STAGE 4: COVID-domain knowledge integration
    STAGE 5: Ensemble consensus
    """
    
    print("\n🔬 RUNNING MAFS ALGORITHM...")
    
    mafs_results = {}
    
    # STAGE 1: Variance & Correlation Filtering
    print("\n📊 STAGE 1: Variance & Correlation Filtering")
    from sklearn.feature_selection import VarianceThreshold
    
    var_selector = VarianceThreshold(threshold=0.01)
    X_var = var_selector.fit_transform(X)
    stage1_features = [f for i, f in enumerate(feature_names) if var_selector.get_support()[i]]
    print(f"   After variance filtering: {len(stage1_features)} features")
    
    mafs_results['stage1'] = stage1_features
    
    # STAGE 2: Statistical Significance
    print("\n📈 STAGE 2: Statistical Significance Testing")
    X_stage1 = X[stage1_features]
    
    f_scores, p_values = f_classif(X_stage1, y)
    stage2_features = [f for f, p in zip(stage1_features, p_values) if p < 0.05]
    print(f"   After significance testing: {len(stage2_features)} features (p < 0.05)")
    
    mafs_results['stage2'] = stage2_features
    
    # STAGE 3: ML Stability with Cross-Validation
    print("\n🤖 STAGE 3: ML Importance with Stability")
    if len(stage2_features) > 0:
        X_stage2 = X[stage2_features]
        
        feature_importance_cv = []
        for train_idx, val_idx in StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE).split(X_stage2, y):
            X_tr = X_stage2.iloc[train_idx]
            y_tr = y.iloc[train_idx]
            
            rf_cv = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
            rf_cv.fit(X_tr, y_tr)
            
            fold_importance = dict(zip(stage2_features, rf_cv.feature_importances_))
            feature_importance_cv.append(fold_importance)
        
        # Calculate stability (low CV = stable)
        stable_features = []
        for feature in stage2_features:
            fold_scores = [fold_imp[feature] for fold_imp in feature_importance_cv]
            mean_importance = np.mean(fold_scores)
            cv_coefficient = np.std(fold_scores) / (mean_importance + 1e-8)
            
            if mean_importance > 0.01 and cv_coefficient < 0.5:
                stable_features.append(feature)
        
        stage3_features = stable_features
        print(f"   After stability analysis: {len(stage3_features)} features")
    else:
        stage3_features = stage2_features
    
    mafs_results['stage3'] = stage3_features
    
    # STAGE 4: COVID Domain Knowledge Integration
    print("\n🏥 STAGE 4: COVID-Domain Knowledge Integration")
    covid_relevant_keywords = ['dose', 'vaccine', 'covid', 'immunotherapy', 
                              'allergic', 'chronic', 'test']
    
    domain_filtered = [f for f in stage3_features 
                      if any(kw in f.lower() for kw in covid_relevant_keywords)]
    
    if len(domain_filtered) == 0:
        domain_filtered = stage3_features  # Fallback to stage3 if no COVID keywords
    
    stage4_features = domain_filtered
    print(f"   After domain filtering: {len(stage4_features)} features")
    
    mafs_results['stage4'] = stage4_features
    
    # STAGE 5: Ensemble Consensus
    print("\n🎯 STAGE 5: Ensemble Consensus")
    print(f"   Final MAFS selected: {len(stage4_features)} features")
    
    mafs_results['final'] = stage4_features
    
    return mafs_results

mafs_results = mafs_algorithm(X_full, y, X_full.columns.tolist())

mafs_selected = mafs_results['final']
print(f"\n📌 MAFS ALGORITHM RESULTS: {len(mafs_selected)} features")
print(f"Features: {mafs_selected}")

# ============================================================================
# STEP 4: FEATURE SET DECISION (CRITICAL DECISION POINT)
# ============================================================================

print("\n" + "="*80)
print("STEP 4: FEATURE SET DECISION - COMBINING TRADITIONAL + MAFS")
print("="*80)

print(f"\n📊 COMPARING APPROACHES:")
print(f"   Traditional (≥2 methods): {len(traditional_selected)} features")
print(f"   MAFS Novel Algorithm: {len(mafs_selected)} features")

# Combine using UNION (most comprehensive)
final_features = list(set(traditional_selected + mafs_selected))
final_features.sort()

print(f"\n🎯 DECISION: USING UNION APPROACH")
print(f"   Combined set: {len(final_features)} features")
print(f"   Features: {final_features}")

print(f"\n📋 RATIONALE:")
print(f"   ✅ Traditional methods: Consensus-based, established methodology")
print(f"   ✅ MAFS novel: Adaptive, domain-aware, stability-focused")
print(f"   ✅ Union: Comprehensive coverage, complementary approaches")
print(f"   ✅ Reduced from {X_full.shape[1]} to {len(final_features)} features")

# Store decision metadata
FEATURE_SELECTION_METADATA = {
    'total_original_features': X_full.shape[1],
    'traditional_count': len(traditional_selected),
    'mafs_count': len(mafs_selected),
    'final_count': len(final_features),
    'approach': 'Union (Traditional + MAFS)',
    'final_features': final_features
}

# ============================================================================
# STEP 5: DATA PREPROCESSING
# ============================================================================

print("\n" + "="*80)
print("STEP 5: DATA PREPROCESSING")
print("="*80)

# Prepare final feature set
X = X_full[final_features].copy()
print(f"✅ Selected features shape: {X.shape}")

# Train-Test Split (BEFORE SMOTE - correct methodology)
print("\n📊 TRAIN-TEST SPLIT (80-20 stratified):")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"   Training set: {X_train.shape}")
print(f"   Test set: {X_test.shape}")
print(f"   Train target distribution: {y_train.value_counts().to_dict()}")
print(f"   Test target distribution: {y_test.value_counts().to_dict()}")

# Apply SMOTE ONLY to training data (correct methodology)
print("\n⚙️ APPLYING SMOTE (Class Imbalance Handling - TRAIN ONLY):")
smote = SMOTE(random_state=RANDOM_STATE)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"   Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"   After SMOTE: {pd.Series(y_train_balanced).value_counts().to_dict()}")
print(f"   Training set after SMOTE: {X_train_balanced.shape}")

# ============================================================================
# STEP 6: MODEL TRAINING & EVALUATION
# ============================================================================

print("\n" + "="*80)
print("STEP 6: MODEL TRAINING & EVALUATION")
print("="*80)

# Define models with consistent configurations
MODELS_CONFIG = {
    "Logistic Regression": LogisticRegression(max_iter=1000, solver='lbfgs', random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=7, min_samples_leaf=6, 
        min_samples_split=8, class_weight='balanced', random_state=RANDOM_STATE
    ),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                            eval_metric='logloss', random_state=RANDOM_STATE)
}

print(f"\n🤖 TRAINING {len(MODELS_CONFIG)} MODELS...")

model_results = []

for model_name, model in MODELS_CONFIG.items():
    print(f"\n   Training {model_name}...")
    
    # Train on balanced data
    model.fit(X_train_balanced, y_train_balanced)
    
    # Predictions
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    if y_proba is not None:
        auc = roc_auc_score(y_test, y_proba)
    else:
        auc = 0.0
    
    model_results.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'AUC': auc,
        'object': model,
        'predictions': y_pred,
        'probabilities': y_proba
    })
    
    print(f"      Accuracy: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")
    print(f"      F1-Score: {f1:.4f} | AUC-ROC: {auc:.4f}")

# Create results DataFrame
df_model_results = pd.DataFrame(model_results)
df_model_results_display = df_model_results[['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'AUC']].copy()

print("\n" + "="*80)
print("📊 MODEL PERFORMANCE SUMMARY")
print("="*80)
print(df_model_results_display.to_string(index=False))

# Select best model
df_ranked = df_model_results.sort_values('F1', ascending=False)
BEST_MODEL = df_ranked.iloc[0]['Model']
BEST_MODEL_OBJECT = df_ranked.iloc[0]['object']

print(f"\n🏆 BEST MODEL: {BEST_MODEL}")
print(f"   F1-Score: {df_ranked.iloc[0]['F1']:.4f}")
print(f"   Accuracy: {df_ranked.iloc[0]['Accuracy']:.4f}")

print("\n✅ MODEL TRAINING COMPLETED!")

# Store all important variables
MODEL_RESULTS_METADATA = {
    'best_model_name': BEST_MODEL,
    'best_model_object': BEST_MODEL_OBJECT,
    'all_results': df_model_results,
    'results_display': df_model_results_display
}

print("\n" + "="*80)
print("🎓 CLEAN END-TO-END ANALYSIS COMPLETED")
print("="*80)
print(f"\n✅ FINAL SUMMARY:")
print(f"   Original features: {FEATURE_SELECTION_METADATA['total_original_features']}")
print(f"   Final features: {FEATURE_SELECTION_METADATA['final_count']}")
print(f"   Selection approach: {FEATURE_SELECTION_METADATA['approach']}")
print(f"   Best model: {BEST_MODEL}")
print(f"   Best F1-Score: {df_ranked.iloc[0]['F1']:.4f}")
print(f"\n🚀 Ready for Q1 publication components!")

🎓 OXFORD UNIVERSITY PhD-LEVEL ANALYSIS
COVID-19 VACCINE SIDE EFFECTS PREDICTION - CLEAN SYSTEMATIC CODE

STEP 1: DATA LOADING & CLEANING
✅ Loaded raw data: (395, 27)
✅ Standardized 27 column names
✅ Target variable: Side_effects_of__COVID-19_vaccine
✅ Retained 27 numeric features + target
✅ Missing values: 0
✅ Features shape: (395, 26)
✅ Target shape: (395,)
✅ Target distribution: {1: 262, 0: 133}

STEP 2: TRADITIONAL FEATURE SELECTION

📊 APPLYING FEATURE SELECTION METHODS:

1️⃣ CHI-SQUARE TEST:
   ✅ Calculated scores for 26 features

2️⃣ MUTUAL INFORMATION:
   ✅ Calculated scores for 26 features

3️⃣ RANDOM FOREST IMPORTANCE:
   ✅ Calculated importance for 26 features

4️⃣ BORUTA ALGORITHM:
   ✅ Boruta selected 1 features

📋 TRADITIONAL FEATURE SELECTION RESULTS:
   Chi2 selected: 7 features
   MI selected: 8 features
   RF selected: 13 features
   Boruta selected: 1 features
   📌 CONSENSUS (≥2 methods): 6 features
   Features: ['Region', 'allergic_reaction', 'believe_vaccines_safe', 

ImportError: cannot import name 'f_classif' from 'scipy.stats' (e:\Covid19\.venv\Lib\site-packages\scipy\stats\__init__.py)